# 09 — Loss

Hands-on companion to [`docs/09-loss.md`](../docs/09-loss.md).

A loss turns an outcome into one number that tells a model how to improve. We will first trace MSE on ordinary numbers, then follow cross-entropy through a tiny next-word prediction—one vector operation at a time.

> Run this with the notebook extra installed: `pip install -e ".[notebooks]"`

In [1]:
import numpy as np

from bonsaigrad import Leaf
from bonsaigrad.nn import CrossEntropyLoss, MSELoss

## MSE: one error per number

For a numeric prediction, each entry has a target value. MSE squares each error, then averages those squared errors. Backpropagation tells every prediction whether it should rise or fall.

In [2]:
predictions = Leaf([1.0, 2.0, 4.0])
targets = np.array([1.0, 4.0, 3.0])
mse = MSELoss()(predictions, targets)
mse.wire()

print(f"predictions: {predictions.data}")
print(f"targets:     {targets}")
print(f"errors:      {predictions.data - targets}")
print(f"MSE:         {mse.data:.3f}")
print(f"gradients:   {predictions.grad}")

predictions: [1. 2. 4.]
targets:     [1. 4. 3.]
errors:      [ 0. -2.  1.]
MSE:         1.667
gradients:   [ 0.         -1.33333333  0.66666667]


## Cross-entropy: one choice among competing words

Now each row is a next-word prediction. The final axis contains three possible words. The target says that `cat` was the word that actually appeared at both positions.

The first row is badly wrong: `dog` has the largest score. The second row is nearly right: `cat` already has the largest score.

In [3]:
words = np.array(["cat", "dog", "tree"])
logit_values = np.array([
    [2.0, 8.0, 1.0],  # position 0: dog has the highest score
    [5.0, 1.0, 0.0],  # position 1: cat has the highest score
])
targets = np.array([0, 0])  # cat is word 0 at both positions

print("words:           ", words)
print("logits shape:    ", logit_values.shape)
print(logit_values)
print("targets shape:   ", targets.shape)
print(targets)

words:            ['cat' 'dog' 'tree']
logits shape:     (2, 3)
[[2. 8. 1.]
 [5. 1. 0.]]
targets shape:    (2,)
[0 0]


### Step 1: gather the score of what actually happened

`targets` supplies one word index per row. The coordinates select `cat` from the first row and `cat` from the second. Nothing has been normalised yet: these are still raw logits.

In [4]:
logits = Leaf(logit_values)
coordinates = (*np.indices(targets.shape), targets)
correct_logits = logits[coordinates]

print("coordinates:      ", coordinates)
print("correct logits:  ", correct_logits.data)
print("shape:           ", correct_logits.data.shape)

coordinates:       (array([0, 1]), array([0, 0]))
correct logits:   [2. 5.]
shape:            (2,)


### Step 2: measure the competition within each row of word scores

A correct logit is only meaningful relative to the other possible words. `logsumexp(axis=-1)` reduces the word-score axis, leaving one normalizer for each row.

In [5]:
normalizers = logits.logsumexp(axis=-1)

print("normalizers:     ", np.round(normalizers.data, 3))
print("shape:           ", normalizers.data.shape)
print("row 0 is governed by dog's score of 8; row 1 is governed by cat's score of 5.")

normalizers:      [8.003 5.025]
shape:            (2,)
row 0 is governed by dog's score of 8; row 1 is governed by cat's score of 5.


### Step 3: turn the scores into probabilities

`normalizers` is the log of the total unnormalised score in each row: `log(sum(exp(x_j)))`.

For one class score `x_i`, the subtraction makes `log_probability_i = x_i - log(sum(exp(x_j)))`.

Exponentiating that result gives `exp(x_i) / sum(exp(x_j))`: the usual softmax probability.

In [6]:
log_probabilities = logits - normalizers[:, None]
probabilities = log_probabilities.exp()

print("log probabilities:")
print(np.round(log_probabilities.data, 3))
print("\nprobabilities:")
for position, row in enumerate(probabilities.data):
    print(f"position {position}: " + ", ".join(f"{word}={probability:.3%}" for word, probability in zip(words, row)))
print("row sums:         ", probabilities.data.sum(axis=-1))

log probabilities:
[[-6.003e+00 -3.000e-03 -7.003e+00]
 [-2.500e-02 -4.025e+00 -5.025e+00]]

probabilities:
position 0: cat=0.247%, dog=99.662%, tree=0.091%
position 1: cat=97.556%, dog=1.787%, tree=0.657%
row sums:          [1. 1.]


### Step 4: compare the correct score with its competition

For each row, cross-entropy subtracts the gathered correct logit from that row's normalizer. The first loss is large because `cat` scored `2` while `dog` scored `8`. The second is small because `cat` already dominates its row. The final mean gives both positions equal influence.

In [7]:
losses = normalizers - correct_logits
loss = losses.mean()
library_loss = CrossEntropyLoss()(logit_values, targets)

print("loss per position:", np.round(losses.data, 3))
print(f"mean loss:        {loss.data:.3f}")
print(f"CrossEntropyLoss: {library_loss.data:.3f}")

np.testing.assert_allclose(loss.data, library_loss.data)

loss per position: [6.003 0.025]
mean loss:        3.014
CrossEntropyLoss: 3.014


### Step 5: send a correction back to every word score

The normalizer branch spreads each row's probability across all possible words. The gathered-target branch subtracts one from `cat`. Finally, the mean divides both rows' contributions by two.

So the wrong word `dog` receives a positive gradient in the first row, which gradient descent will lower. The correct word `cat` receives a negative gradient, which gradient descent will raise.

In [8]:
loss.wire()
one_hot_targets = np.eye(len(words))[targets]
expected_gradients = (probabilities.data - one_hot_targets) / len(targets)

print("softmax contribution:")
print(np.round(probabilities.data / len(targets), 3))
print("\ntarget correction:")
print(-one_hot_targets / len(targets))
print("\nlogit gradients:")
print(np.round(logits.grad, 3))

np.testing.assert_allclose(logits.grad, expected_gradients)
print("positive: lower this logit; negative: raise this logit.")

softmax contribution:
[[0.001 0.498 0.   ]
 [0.488 0.009 0.003]]

target correction:
[[-0.5 -0.  -0. ]
 [-0.5 -0.  -0. ]]

logit gradients:
[[-0.499  0.498  0.   ]
 [-0.012  0.009  0.003]]
positive: lower this logit; negative: raise this logit.


## Why the normalizer matters

A raw correct logit cannot tell us whether the model made a good prediction. In both rows below, `cat` has raw score `8`. Cross-entropy still distinguishes them because it also considers every competing word score.

In [9]:
comparison_logits = np.array([[8.0, 2.0, 1.0], [8.0, 20.0, 1.0]])
comparison_targets = np.array([0, 0])
comparison_losses = -np.log(np.exp(comparison_logits - comparison_logits.max(axis=-1, keepdims=True)) / np.exp(comparison_logits - comparison_logits.max(axis=-1, keepdims=True)).sum(axis=-1, keepdims=True))[np.arange(2), comparison_targets]

for row, row_loss in zip(comparison_logits, comparison_losses):
    print(f"logits {row}  →  correct logit {row[0]:.0f}, loss {row_loss:.3f}")

logits [8. 2. 1.]  →  correct logit 8, loss 0.003
logits [ 8. 20.  1.]  →  correct logit 8, loss 12.000


The second row scores cat at 8, but it scores dog at 20: it is confidently wrong.

The forward pass gathers the score of the observed word and compares it with the other possible words. The backward pass then raises the observed word's score and lowers alternatives in proportion to how much probability they received. That is the practical loop cross-entropy gives a classifier.